## Tools

In LangChain, tools are modular, callable functions that allow Large Language Models (LLMs) to interact with external systems—effectively acting as "plugins" to extend the model's capabilities beyond simple text generation.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b", api_key=os.getenv("GROQ_API_KEY"))
response = model.invoke("Why do parrots talk")
print(response)

content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots and their ability to mimic sounds.\n\nFirst, I remember that parrots are part of the bird family that includes cockatiels, macaws, and budgies. They\'re known for their colorful feathers and talking abilities. But why do they do that? I think it\'s related to mimicry. They can imitate human speech, but why would they evolve that trait?\n\nMaybe it has to do with communication in the wild. In their natural habitats, parrots live in social groups. They might use vocalizations to communicate with each other, maybe for things like calling to each other, warning of predators, or establishing territory. If they can mimic sounds, including human speech, it could be an extension of this natural behavior.\n\nI\'ve heard that some birds have a structure in their brains that allows them to learn and reproduce sounds. The syrinx is the vocal organ in birds, and I think parrots have a

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """get the weather at a given location"""
    return f"The weather in {location} is sunny"


model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified New York, I need to call that function with "New York" as the location. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': 'srh7fdszk', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 155, 'total_tokens': 250, 'completion_time': 0.127643081, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.006030668, 'prompt_tokens_details': None, 'queue_time': 0.043549041, 'total_time': 0.133673749}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id=

### tool execution loops

In [8]:
# step 1: Model generates tool calls
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with generated args
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


# Step 3: Pass the results to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in New York is currently sunny. ☀️ Let me know if you'd like additional details!
